In [1]:
import sys
from pathlib import Path

# add the parent directory as a package path 
parent_dir = Path(__file__).resolve().parents[1] if '__file__' in locals() else Path('.').resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

from src.graph_utils import create_layered_graph, assign_properties
from src.lp import standard_max_flow

layer_sizes = [1,2,2,1]
G, layers = create_layered_graph(layer_sizes)
G = assign_properties(G, layer_sizes)
result, flow_dict = standard_max_flow(G)
result['Problem']()['Lower bound']


3.0

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from tqdm import trange

import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib as mpl

import pyomo.environ as pyo
import networkx as nx
import pandas as pd
import numpy as np
import re

class Objectives:
    @staticmethod
    def max_flow(m):
        """primal"""
        return pyo.Objective(expr=m.F, sense=pyo.maximize)

    @staticmethod
    def min_cut_flow(m):
        return pyo.Objective(expr=sum(m.cap[u,v]*m.theta[u,v] for u, v in m.edges), sense=pyo.minimize)

    @staticmethod
    def min_cost_flow(m):
        expr = sum(m.cost[u, v] * m.x[u, v] for (u, v) in m.edges)
        return pyo.Objective(expr=expr, sense=pyo.minimize)

    @classmethod
    def _apply(cls, problem_type, model):

        registry = {
            'max_flow': cls.max_flow,
            'min_cost_flow': cls.min_cost_flow,
        }
        if problem_type not in registry:
            raise ValueError(f"Unknown problem type: {problem_type}")
            
        model.obj = registry[problem_type](model)

class Rules:
    @staticmethod
    def edge_cap_rule(m, u, v):
        """Capacity can be bandwidth, time-requirement, or 1 if assessing count of independent paths"""
        return m.x[u, v] <= m.capacity[u, v] * m.node_state[u] * m.node_state[v] * m.edge_state[u,v]
     
    @staticmethod
    def edge_flow_rule(m, n):
        """Flow out = Flow in for all nodes; originator demand=-1, endpoint demand=1"""
        if n in (m.s, m.t):
            return pyo.Constraint.Skip
        inflow = sum(m.x[u, v] for (u, v) in m.edges if v == n)
        outflow = sum(m.x[u, v] for (u, v) in m.edges if u == n)
        return inflow == outflow
    
    @staticmethod
    def source_flow_rule(m):
        outflow = sum(m.x[u, v] for (u, v) in m.edges if u == m.s)
        inflow = sum(m.x[u, v] for (u, v) in m.edges if v == m.s)
        return outflow - inflow == m.F
    
    @staticmethod
    def sink_flow_rule(m):
        inflow = sum(m.x[u, v] for (u, v) in m.edges if v == m.t)
        outflow = sum(m.x[u, v] for (u, v) in m.edges if u == m.t)
        return inflow - outflow == m.F

    @staticmethod
    def cutset_rule(m, u, v):
        return m.d_resflow[u,v] + m.d_cutset_ind[u] - m.d_cutset_ind[v] >= 0

    @staticmethod
    def st_boundary_rule(m):
        return m.alpha_s - m.alpha_t >= 1

class LP:
    def __init__(self, G):
        self.g = G.copy()
        self.r = Rules()
        self.g = self.__configure_graph()
        self._create_primal_model()

    @staticmethod
    def _return_domain(relaxed: bool):
        return pyo.NonNegativeReals if relaxed else pyo.Binary

    def _create_base_model(self):
        model = pyo.ConcreteModel()
        model.nodes = pyo.Set(initialize=list(self.g.nodes()))
        model.edges = pyo.Set(initialize=list(self.g.edges()), dimen=2)
        # used for monte-carlo simulation
        model.node_state = pyo.Param(model.nodes, within=pyo.Binary, mutable=True, initialize=1)
        model.edge_state = pyo.Param(model.edges, within=pyo.Binary, mutable=True, initialize=1)
        model.s = 's'
        model.t = 't'
        return model
    
    def _create_primal_model(self, relaxed=False):
        model = self._create_base_model()
        domain = self._return_domain(relaxed)
        self.model = model

        # Initial primal variables, params, and constriants 
        model.x = pyo.Var(model.edges, within=domain)
        model.capacity = pyo.Param(model.edges,initialize={(u, v): self.g.edges[u, v]['capacity'] for u, v in self.g.edges()})
        model.cost = pyo.Param(model.edges, initialize={(u,v): self.g.edges[u,v]['cost'] for u, v in self.g.edges()})
        model.demand = pyo.Param(model.nodes,initialize={n: self.g.nodes[n]['demand'] for n in self.g.nodes()})
        model.cap = pyo.Constraint(model.edges, rule=self.r.edge_cap_rule)
        model.flow = pyo.Constraint(model.nodes, rule=self.r.edge_flow_rule)
        if relaxed:
            model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

    def _create_dual_model(self):
        model = self._create_base_model()
        self.dual_model = model 
        
        model.theta = pyo.Var(model.edges, within=pyo.NonNegativeReals) # ST-cut indicator variable
        model.alpha = pyo.Var(model.nodes, within=pyo.NonNegativeReals) # ST-cut forcing variable
        model.alpha_s = 'alpha_s'
        model.alpha_t = 'alpha_t'



    def solve_model(self, problem='max_flow', solver_str='appsi_highs'):
        if problem == 'max_flow':
            self.model.F = pyo.Var(within=pyo.NonNegativeReals)
            self.model.source = pyo.Constraint(rule=self.r.source_flow_rule)
            self.model.sink = pyo.Constraint(rule=self.r.sink_flow_rule)
            Objectives._apply('max_flow', self.model)
            return self._solve_model(solver_str)
    
        if problem == 'min_cost_flow':
            ...
        if problem == 'k_shortest_paths':
            ...        

    def _solve_model(self, solver_str):
        solver = pyo.SolverFactory(solver_str)
        result = solver.solve(self.model)
        return result

    def __configure_graph(self, source='s', sink='t', artifical_source='dS', artifical_sink='dT'):
        """
        Standardizes the graph for LPs including:
        1. Creating super-source/sink artifical nodes
        2. Standardizing cost/capacity parameters for edges
        3. Standardizing demand parameter for nodes 
        4. Create an artifical source/sink to preserve demand balance 
        """
        H = self.g.copy()
        if not H.is_directed():
            H = H.to_directed()

        for _,_,d in H.edges(data=True):
            d.setdefault('capacity', 1)
            d.setdefault('cost', 0)

        for n, data in H.nodes(data=True):
            data.setdefault('demand', 0)

        # Add super source/sink
        H.add_node(source, demand=0, layer=-1)
        H.add_node(sink, demand=0, layer=1+max(d['layer'] for _,d in H.nodes(data=True)))
        supply = [n for n in H.nodes if H.nodes[n].get('demand') < 0]
        demand = [n for n in H.nodes if H.nodes[n].get('demand') > 0]
        for n in supply:
            H.add_edge(source, n, capacity=-H.nodes[n]['demand'], cost=0, artificial=True)
        for n in demand:
            H.add_edge(n, sink, capacity=H.nodes[n]['demand'], cost=0, artificial=True)

        # Normalize demand balance
        demand_imbalance = sum(-H.nodes[n]['demand'] for n in supply) - sum(H.nodes[n]['demand'] for n in demand)
        if demand_imbalance > 0:
            H.add_node(artifical_sink, demand=0, artifical=True, layer=-1)
            H.add_edge(source, artifical_sink, capacity=demand_imbalance, cost=0, artifical=True)
            H.add_edge(artifical_sink, sink, capacity=demand_imbalance, cost=0, artifical=True)
        elif demand_imbalance < 0:
            H.add_node(artifical_source, demand=0, artifical=True, layer=1+max(d['layer'] for _,d in H.nodes(data=True)))
            H.add_edge(artifical_source, sink, capacity=-demand_imbalance, cost=0, artifical=True)
            H.add_edge(source, artifical_source, capacity=-demand_imbalance, cost=0, artifical=True)
        return H

class LP_Solver:

    def __init__(self, lp):
        self.lp = lp
        self.model = lp.model
        self.g = lp.g
        self.solver_result = lp.solve_model()
        self.results = self._build_results()

    def draw_modified_graph(self):
        pos = nx.art_layout(self.g)
        nx.draw(self.g, pos=pos, with_labels=True, node_color='lightblue', 
                node_size=2000, font_size=16, font_weight='bold', arrowsize=20
        )

    def get_residual_graph(self):
        R = nx.DiGraph()
        for u,v in self.model.edges:
            capacity = pyo.value(self.model.capacity[u,v])
            flow = pyo.value(self.model.x[u,v])
            forward = capacity - flow
            reverse = flow
            if forward > 0:
                R.add_edge(u,v,residual=forward)
            if reverse > 0:
                R.add_edge(v,u, residual=reverse)
        return R  

    def _get_edge_results(self):
        edge_results = {}
        for u, v in self.model.edges:
            flow = pyo.value(self.model.x[u, v])
            capacity = pyo.value(self.model.capacity[u, v])
            cost = pyo.value(self.model.cost[u, v])
            edge_results[(u, v)] = {
                'flow': flow, 
                'capacity': capacity, 
                'residual': capacity - flow,
                'utilization': (flow / capacity if capacity > 0 and capacity != float('inf') else 0),
                'cost': cost,
                'total_cost': flow * cost,
                'artificial': self.g.edges[u, v].get('artificial', False)
            }
        return edge_results

    def _get_node_results(self):
        node_results = {}
        for n, d in self.g.nodes(data=True):
            demand = d.get('demand', 0)
            node_type = 'supply' if demand < 0 else 'demand' if demand > 0 else 'transshipment'
            inflow = sum(pyo.value(self.model.x[u, n]) for u in self.g.nodes if (u,n) in self.model.edges)
            outflow = sum(pyo.value(self.model.x[n,v]) for v in self.g.nodes if (n,v) in self.model.edges)
            
            if (self.model.s, n) in self.model.edges:
                satisfied = pyo.value(self.model.x[self.model.s, n]) 
            elif (n, self.model.t) in self.model.edges:
                satisfied = pyo.value(self.model.x[n, self.model.t])
            else:
                satisfied = 0

            node_results[n] = {
                'node_type': node_type,
                'inflow': inflow,
                'outflow': outflow,
                'demand': demand,
                'remaining_supply': -(demand + inflow - outflow) if node_type == 'supply' else 0,
                'demand_satisfied': satisfied,
                'satisfaction_rate': abs(satisfied/demand) if demand != 0 else 0, 
                'cutset_group': self._get_dual(self.model.flow[n]) if n not in ('s', 't') else None 
            }
        return node_results 

    def _get_bottlenecks(self):
        if not hasattr(self.model, 'dual'):
            return None
        bottlenecks = []
        for edge in self.model.edges:
            constraint = self.model.cap[edge]
            dual = self.model.dual.get(constraint)
            if dual is not None and dual > 0:
                bottlenecks.append(edge)
        return bottlenecks

    def _get_dual(self, constraint):
        return None if not hasattr(self.model, 'dual') else self.model.dual.get(constraint)

    def _build_results(self):
        solver_result = self.solver_result
        results = {
            'problem': {
                'lower_bound': solver_result['Problem']()['Lower bound'],
                'upper_bound': solver_result['Problem']()['Upper bound'],
                'obj_function_count': self.model.nobjectives(),
                'constraint_count': self.model.nconstraints(),
                'variable_count': self.model.nvariables(),
                'sense': str(self.model.obj.sense),
            },
            'solver': {
                'status': str(solver_result['Solver']()['Status']),
                'termination_condition': str(solver_result['Solver']()['Termination condition']),
                'termination_message': str(solver_result['Solver']()['Termination message']),
            }
        }
        node_results = self._get_node_results()
        edge_results = self._get_edge_results()
        supply_nodes = [r for r in node_results.values() if r['node_type'] == 'supply']
        demand_nodes = [r for r in node_results.values() if r['node_type'] == 'demand']
        total_supply = sum(-r['demand'] for r in supply_nodes)
        total_used_supply = sum(r['demand_satisfied'] for r in supply_nodes)
        total_demand = sum(r['demand'] for r in demand_nodes)
        total_satisfied = sum(r['demand_satisfied'] for r in demand_nodes)
        total_unmet = total_demand - total_satisfied

        results['node_results'] = node_results
        results['edge_results'] = edge_results
        results['metrics'] = {
            'total_supply': total_supply,
            'total_used_supply': total_used_supply,
            'total_unused_supply': (total_supply - total_used_supply),
            'total_demand': total_demand,
            'total_satisfied': total_satisfied,
            'total_unmet': total_unmet,
            'bottlenecks': self._get_bottlenecks(),
            'total_satisfaction_rate': (total_satisfied / total_demand if total_demand > 0 else None),
        }
        return results

class GraphDrawer:
    def __init__(self, solver):
        self.solver = solver
        self.lp = solver.lp
        self.g = self.lp.g
        self.model = self.lp.model
        self.node_results = solver.results['node_results']
        self.bgar_cmap = mcolors.LinearSegmentedColormap.from_list("edge_cmap", plt.cm.Blues(np.linspace(0.2, 0.7, 256)))

    def draw(self, mode="flow", pos=None, figsize=(24, 8), **kwargs):
        fig, ax = plt.subplots(figsize=figsize)
        if pos is None:
            pos = nx.multipartite_layout(self.g, subset_key="layer")

        node_data = self.solver._get_node_results()
        edge_data = self.solver._get_edge_results()

        cmap = plt.cm.Reds
        cmap = mcolors.LinearSegmentedColormap.from_list("edge_cmap", cmap(np.linspace(0.15, 0.8, 256)))

        if mode == "flow":
            self._draw_flow(pos, node_data, edge_data, cmap, ax=ax, **kwargs)
        elif mode == "residual":
            self._draw_residual(pos, node_data, edge_data, ax=ax, cmap=cmap, **kwargs)
        elif mode == "demand":
            self._draw_demand(pos, node_data, edge_data, ax=ax, **kwargs)
        elif mode == "cost":
            self._draw_cost(pos, node_data, edge_data, cmap, ax=ax, **kwargs)
        else:
            raise ValueError(f"Unknown drawing mode: {mode}")

        ax.set_axis_off()
        plt.show()

    def _get_node_colors(self, node_data):
        node_colors = []
        for n, data in self.node_results.items():
            if data['node_type'] != 'transshipment':
                rate = max(0.0, min(1.0, data["satisfaction_rate"]))
                node_colors.append(self.bgar_cmap(rate))
            else:
                node_colors.append("#aaaaaa")#gray
        return node_colors

    def _add_node_colorbar(self, ax):
        """Adds a dedicated colorbar for Node Satisfaction on the left."""
        norm_node = mpl.colors.Normalize(vmin=0, vmax=1)
        sm_node = mpl.cm.ScalarMappable(norm=norm_node, cmap=self.bgar_cmap)
        sm_node.set_array([])
        cbar_node = ax.figure.colorbar(sm_node, ax=ax, location="left", pad=0.08, shrink=0.8)
        cbar_node.set_label("Node Satisfaction Rate")

    def _draw_flow(self, pos, node_data, edge_data, cmap, ax=None, **kwargs):
        if ax is None:
            fig, ax = plt.subplots()

        edge_utilization = []
        edge_widths = []
        edge_colors = []
        flow_cmap = cmap
        for u, v in self.g.edges:
            data = edge_data[(u, v)]
            utilization = data["utilization"]
            edge_utilization.append(utilization)
            edge_widths.append(1 + 2 * utilization)
            if utilization > 0:
                edge_colors.append(flow_cmap(utilization))
            else: 
                edge_colors.append('#cccccc')

        node_colors = self._get_node_colors(node_data)

        nx.draw_networkx_nodes(
            self.g, pos, node_color=node_colors, node_size=500, ax=ax
        )
        nx.draw_networkx_edges(
            self.g,pos,width=edge_widths,arrows=True,edge_color=edge_colors,edge_cmap=cmap,ax=ax,)
        nx.draw_networkx_labels(self.g, pos, ax=ax)

        # Edge Colorbar (Right)
        norm_edge = mpl.colors.Normalize(vmin=0, vmax=1)
        sm_edge = mpl.cm.ScalarMappable(norm=norm_edge, cmap=cmap)
        sm_edge.set_array([])
        cbar_edge = ax.figure.colorbar(sm_edge, ax=ax, location="right", pad=0.02, shrink=0.8)
        cbar_edge.set_label("Capacity Utilization")

        # Node Colorbar (Left)
        self._add_node_colorbar(ax)
        ax.set_title("Maximum Flow w/ Highlighted Originator & Endpoint Signaling")
        
        legend_thicknesses = [1.0, 3.0, 5.0]
        legend_handles = [
            mlines.Line2D([],[],color="black",linewidth=w,label=f"Flow: {w}",)for w in legend_thicknesses
        ]
        plt.legend(handles=legend_handles,title="Quantity of Flow",loc="lower right",frameon=True,)

    def _draw_demand(self, pos, node_data, edge_data, ax=None, **kwargs):
        if ax is None:
            fig, ax = plt.subplots()

        node_colors = self._get_node_colors(node_data)

        nx.draw_networkx_nodes(
            self.g, pos, node_color=node_colors, node_size=500, ax=ax
        )
        nx.draw_networkx_edges(self.g, pos, arrows=True, ax=ax)

        labels = {}
        for n in self.g.nodes:
            data = node_data.get(n, {})
            if data.get("type") == "demand":
                labels[n] = (
                    f"{n}\n"
                    f"D: {data.get('demand', 0):.1f}\n"
                    f"S: {data.get('satisfied', 0):.1f}\n"
                    f"U: {data.get('unmet', 0):.1f}"
                )
            else:
                labels[n] = str(n)

        nx.draw_networkx_labels(self.g, pos, labels=labels, ax=ax)
        self._add_node_colorbar(ax)

    def _draw_residual(self, pos, node_data, edge_data, ax=None, **kwargs):
        if ax is None:
            fig, ax = plt.subplots()
        R = self.lp.get_residual_graph()
        active_edges = [(u, v) for u, v in R.edges if R[u][v]['residual'] > 1e-5]
        residuals = [R[u][v]['residual'] for u, v in active_edges]
        max_res = max(residuals, default=1.0)
        widths = [1 + 2.5 * (r / max_res) for r in residuals]
        alphas = [0.3 + 0.7 * (r / max_res) for r in residuals]
        node_colors = ["#8f8f8f"]*len(R.nodes)
        nx.draw_networkx_nodes(R, pos, node_color=node_colors, node_size=500, ax=ax)
        
        # Draw edges with varying alphas
        for idx, (u, v) in enumerate(active_edges):
            nx.draw_networkx_edges(
                R, pos,edgelist=[(u, v)],width=widths[idx],alpha=alphas[idx],arrows=True,ax=ax
            )
            
        nx.draw_networkx_labels(R, pos, ax=ax)
        ax.set_title("Residual Graph")

    def _draw_cost(self, pos, node_data, edge_data, cmap, ax=None, **kwargs):
        if ax is None:
            fig, ax = plt.subplots()

        edge_colors = [edge_data[(u, v)]["total_cost"] for u, v in self.g.edges]
        edge_widths = [
            1 + 2 * edge_data[(u, v)]["utilization"] for u, v in self.g.edges
        ]
        node_colors = self._get_node_colors(node_data)

        nx.draw_networkx_nodes(
            self.g, pos, node_color=node_colors, node_size=500, ax=ax
        )
        nx.draw_networkx_edges(
            self.g,
            pos,
            width=edge_widths,
            edge_color=edge_colors,
            edge_cmap=cmap,
            arrows=True,
            ax=ax,
        )
        nx.draw_networkx_labels(self.g, pos, ax=ax)

        # Edge Colorbar (Right side)
        min_cost, max_cost = min(edge_colors, default=0), max(
            edge_colors, default=1
        )
        norm_edge = mpl.colors.Normalize(vmin=min_cost, vmax=max_cost)
        sm_edge = mpl.cm.ScalarMappable(norm=norm_edge, cmap=cmap)
        sm_edge.set_array([])
        cbar_edge = ax.figure.colorbar(
            sm_edge, ax=ax, location="right", pad=0.02, shrink=0.8
        )
        cbar_edge.set_label("Total Cost")

        # Node Colorbar (Left side)
        self._add_node_colorbar(ax)

class GraphMC:
    def __init__(self, solver):
        self.solver = solver
        self.g = solver.g
        self.model = solver.model

    def _preprocess_network_simulation(self):
        self.nodes = list(self.g.nodes())
        self.edges = list(self.g.edges())
        self.node_p = np.array([self.g.nodes[n].get('probability1', 1) for n in self.nodes])
        self.edge_p = np.array([self.g.edges[n].get('probability', 1) for n in self.edges])

    def _process_result(self, result, i:int):
        flow = pyo.value(self.model.F)    
        node_operational = int(self.node_state.sum())
        node_failed = len(self.nodes) - node_operational
        edge_operational = int(self.edge_state.sum())
        edge_failed = len(self.edges) - edge_operational
        row = {
            'sim_number': i, 'flow': flow, 
            'node_operational': node_operational, 'node_failed': node_failed, 
            'edge_operational': edge_operational, 'edge_failed': edge_failed,
            'node_opr_ratio': node_operational/len(self.nodes), 
            'edge_opr_ratio': edge_operational/len(self.edges),
            'termination_condition': str(result.solver.termination_condition),
            'flow_failure_indicator': 1 if flow == 0 else 0
        }
        for n, state in zip(self.nodes, self.node_state):
            row[f'node_{n}'] = int(state)
        # Edge states
        for (u, v), state in zip(self.edges, self.edge_state):
            row[f'edge_({u},{v})'] = int(state)
        return row

    def _sample_states(self, rng):
        return rng.binomial(1, self.node_p), rng.binomial(1, self.edge_p)

    def stochastic_max_flow(self, n_sim=1000, seed=123, solver_str='appsi_highs'):
        rng = np.random.default_rng(seed)
        pyo_solver = pyo.SolverFactory(solver_str)
        self._preprocess_network_simulation()
        results = []
        # simulation changes the lp-parameters, not the graph nor model
        for i in trange(n_sim):
            self.node_state, self.edge_state = self._sample_states(rng)
            for n, state in zip(self.nodes, self.node_state):
                self.model.node_state[n] = int(state)
            for (u, v), state in zip(self.edges, self.edge_state):
                self.model.edge_state[u, v] = int(state)
            result = pyo_solver.solve(self.model)
            results.append(self._process_result(result, i))
        return pd.DataFrame(results)
    
    @staticmethod
    def _get_influence_logic(df, component_columns):
        results = []
        for component in component_columns:
            flow_operational = df.loc[df[component] == 1, 'flow'].mean()
            flow_failed = df.loc[df[component] == 0, 'flow'].mean()
            results.append({
                'component': component,
                'flow_if_operational': flow_operational,
                'flow_if_failed': flow_failed if not None else 0,
                'flow_difference': flow_operational - flow_failed,
                'n_operational': (df[component] == 1).sum(),
                'n_failed': (df[component] == 0).sum() if not None else 0,
                'observed_performance_rate': (df[component] == 1).sum()/df.shape[0]
            })
        return pd.DataFrame(results).sort_values('flow_difference',ascending=False).reset_index(drop=True)

    def get_node_edge_influence(self, df):
        """ Function for getting additonal insights from stochastic max flow simulation """
        self.node_cols = [c for c in df.columns if re.match(r"\bnode_(\d+)\b", c)]
        self.edge_cols = [c for c in df.columns if re.match(r'^edge_\(\d+,\d+\)$',c)]
        node_df = self._get_influence_logic(df, self.node_cols).fillna(0)
        edge_df = self._get_influence_logic(df, self.edge_cols).fillna(0)
        return node_df, edge_df

    @staticmethod
    def plot_reliability_curve(df, flow_requirement=1):
        flow_values = np.sort(df['flow'])
        reliability = np.array([np.mean(df['flow'] >= f)for f in flow_values])
        plt.figure(figsize=(8, 5))
        plt.plot(flow_values, reliability, label=f'Probabilty G will have x independent paths')
        plt.axvline(flow_requirement,linestyle='--',label=f'Required flow: {flow_requirement}', c='red')
        plt.xlabel('Maximum Flow')
        plt.ylabel('Reliability')
        plt.title('Network Maximum-Flow Reliability Curve', weight='bold')
        plt.ylim(0, 1.02)
        plt.legend(loc='best')
        plt.grid(True, alpha=0.4)
        plt.show()

    def get_node_edge_importance(self, df):
        """ 
        Returns:
        1. A dataframe of fit node/edge importances with respect to flow 
        2. A dataframe of permutation importances and anwsers the question:
                How much does shuffling results of a single node/arc effect the predicted outcome?
        """
        X = df[self.node_cols + self.edge_cols]
        y = df['flow']

        X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=123)
        model = RandomForestRegressor(n_estimators=300,random_state=123,n_jobs=-1)
        model.fit(X_train, y_train)
        # using a permutation model instead --> stratifes successes and failures
        perm = permutation_importance(
            model,X_test,y_test,n_repeats=10,random_state=123,n_jobs=-1
        ) 

        #pred = model.predict(X_test)
        #print("R-squared:", r2_score(y_test, pred))
        #print("MAE:", mean_absolute_error(y_test, pred))

        importance_df = pd.DataFrame(
            {'component': X.columns,'importance': model.feature_importances_}
        ).sort_values('importance',ascending=False)
        perm_imp_df = pd.DataFrame(
            {'component': X.columns,'importance': perm.importances_mean}
        ).sort_values('importance',ascending=False)
        return importance_df, perm_imp_df


In [4]:
lp = LP(G)
lp.model.pprint()

2 Set Declarations
    edges : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :   10 : {(0, 1), (0, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 5), (4, 5), (5, 't'), ('s', 0)}
    nodes : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    8 : {0, 1, 2, 3, 4, 5, 's', 't'}

5 Param Declarations
    capacity : Size=10, Index=edges, Domain=Any, Default=None, Mutable=False
        Key      : Value
          (0, 1) :     5
          (0, 2) :     3
          (1, 3) :     1
          (1, 4) :     5
          (2, 3) :     3
          (2, 4) :     1
          (3, 5) :     1
          (4, 5) :     2
        (5, 't') :     4
        ('s', 0) :     4
    cost : Size=10, Index=edges, Domain=Any, Default=None, Mutable=False
        Key      : Value
          (0, 1) :  33.13050968485109
          (0, 2) : 16.331217716750878
          (1, 3) :  33.30759100085764
         